In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib

# Load the data
df = pd.read_csv('synthetic_triage_data.csv')
print(df.head()) # This will show you the first 5 rows of your data

   temperature  blood_pressure_sys  blood_pressure_dia  heart_rate  \
0         37.2                 120                  80          72   
1         39.5                 145                  95         115   
2         41.0                  90                  50         140   
3         36.8                 118                  78          68   
4         38.0                 130                  85          90   

                          chief_complaint  triage_priority  
0                         routine checkup                2  
1        severe fever and sweating chills                1  
2  unresponsive cold clammy skin no pulse                0  
3                           mild headache                2  
4                    cough and mild fever                2  


In [9]:
# defining features(x) and target(y) tell the model what data to look at (X) and what answer it is trying to predict (y).
x = df[['temperature', 'blood_pressure_sys', 'blood_pressure_dia', 'heart_rate', 'chief_complaint']] #our inputs
y = df[['triage_priority']] #our target ie priority level
#split into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

In [11]:
#Machine learning models can only understand numbers,not text like 
#"severe fever".We use a *ColumnTransformer* to handle the numeric vitals 
#normally, but it uses a TfidfVectorizer(How It WorksTerm Frequency (TF): Counts how often a word appears in a specific document.Inverse Document Frequency (IDF): Reduces the score of common words (like "the" or "is") and increases the score of rare words.Combines both metrics to highlight words that are unique and meaningful to individual texts.) to convert the chief_complaint text
#into a matrix of numbers based on word frequency

#1. Define how to handle the different types of data
preprocessor = ColumnTransformer(
    transformers=[
        ('num','passthrough',['temperature', 'blood_pressure_sys', 'blood_pressure_dia', 'heart_rate']),
        ('text',TfidfVectorizer(),'chief_complaint')
    ]
)
#2. Create the pipeline: First process the data then feed it to the Random Forest model
#A Scikit-Learn Pipeline is exactly the same concept. It guarantees that our data flows in a strict sequence:
#Step 1: Run the data through the Pre-processor (translate text to numbers).
#Step 2: Feed those numbers into the Classifier (the AI model).
#3. What is happening inside the Random Forest Classifier?
#A "Decision Tree" is basically a giant, automatically generated if/else block.
#If Temp > 39 AND heart_rate > 100 AND symptoms contain "fever" -> Priority 1.
#But a single decision tree is prone to making mistakes. A Random Forest is exactly what it sounds like: it generates hundreds of different decision trees, gives them all slightly different parts of the data to look at, and then has them vote on the final answer. It is incredibly powerful and highly accurate for tabular database records.
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])
#3. Train the model
pipeline.fit(x_train, y_train)
print("Model trained successfully!")

/home/charity/PROJECTS/Portfolio/Healthcare-system/ai-triage-service/venv/lib64/python3.14/site-packages/sklearn/base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Model trained successfully!


In [12]:
# Save the trained pipeline as a file
joblib.dump(pipeline, 'triage_model.pkl')
print("Model saved as triage_model.pkl")

Model saved as triage_model.pkl
